# Unit 06 - Randomization Unit (Demo)

**Atoms served:** `U06-A3` (contamination / spillover), `U06-A4` (blocking preview), `U06-A5` (randomization unit - **not covered on video**; Kohavi ch. 14 and this notebook carry it)

**Estimated runtime:** ~25 seconds

**After this notebook you can:** compare standard errors at user, session, and cluster levels, show contamination when the wrong unit is used, and explain why cluster randomization costs power.

## Without code

Read the printed standard errors and contamination counts.

1. Session-level `SE` should be artificially smallest; user-level in the middle; cluster-level largest.
2. Wrong-unit row: some control sessions should appear on treated users (contamination > 0).
3. Power table: required `n` rises as intra-cluster correlation rises.

Same story without running code.

## 1. The question

The same two-week checkout test is analysed three ways: randomise **users**, **sessions**, or **stores** (clusters). Marketplace spillover from `V16` means treated users can affect control users in the same cluster.

Which analysis looks most precise - and is that precision honest?

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

500 users, each with several sessions, grouped into 25 stores (clusters). True user-level `ATE = 0.02` on conversion. Stores share a random effect - high-traffic malls behave alike.

In [ ]:
n_users = 500
n_clusters = 25
users_per_cluster = n_users // n_clusters
cluster_id = np.repeat(np.arange(n_clusters), users_per_cluster)
user_id = np.arange(n_users)
sessions_per_user = np.random.poisson(2, n_users)

# cluster-randomized treatment (50% of clusters treated)
cluster_treat = np.random.binomial(1, 0.5, n_clusters)
D_user = cluster_treat[cluster_id]

cluster_effect = np.random.normal(0, 0.03, n_clusters)[cluster_id]
Y_user = np.random.binomial(1, np.clip(0.15 + 0.02*D_user + cluster_effect, 0, 1))

rows = []
for u in range(n_users):
    for s in range(sessions_per_user[u]):
        rows.append({'user_id': u, 'cluster_id': cluster_id[u], 'D_user': D_user[u],
                     'Y': np.random.binomial(1, Y_user[u]), 'session_id': len(rows)})
sessions = pd.DataFrame(rows)
print('Users:', n_users, 'Sessions:', len(sessions), 'Clusters:', n_clusters)

## 4. The naive move

Analyse at **session level** because that is what the event log contains. More rows, so it must be better.

In [ ]:
def ate_se(df, treat_col, outcome='Y'):
    g = df.groupby(treat_col)[outcome]
    m = g.mean()
    v = g.var()
    n = g.count()
    ate = m[1] - m[0]
    se = np.sqrt(v[1]/n[1] + v[0]/n[0])
    return ate, se

ate_s, se_s = ate_se(sessions, 'D_user')
print('Session-level ATE:', round(ate_s,4), 'SE:', round(se_s,4))

Session-level analysis uses many rows but treats sessions as independent - they are not.

## 5. What actually happens

Compare **user**, **session**, and **cluster** aggregation. Standard errors should diverge.

In [ ]:
user_df = sessions.groupby(['user_id','D_user'])['Y'].mean().reset_index()
cluster_df = sessions.groupby(['cluster_id','D_user'])['Y'].mean().reset_index()

results = []
for name, data, col in [('user', user_df, 'D_user'), ('session', sessions, 'D_user'), ('cluster', cluster_df, 'D_user')]:
    ate, se = ate_se(data, col)
    results.append({'unit': name, 'n_rows': len(data), 'ATE': round(ate,4), 'SE': round(se,4)})
print(pd.DataFrame(results))

Session-level `SE` is artificially small because sessions within a user repeat the same outcome.

**Wrong randomization unit (`U06-A3`).** Randomise sessions but some users see both arms across devices.

In [ ]:
# Simulate session-level randomisation on same users -> contamination
np.random.seed(RANDOM_SEED + 1)
wrong = sessions.copy()
wrong['D_session'] = np.random.binomial(1, 0.5, len(wrong))
contaminated_users = wrong.groupby('user_id')['D_session'].nunique()
n_contam = (contaminated_users > 1).sum()
print('Users seeing BOTH arms under session-level assignment:', n_contam, 'of', n_users)
print('Those users break the control group comparison.')

Session-level assignment lets the same person appear in treatment and control - classic contamination.

**Intra-cluster correlation (`U06-A5`).** Cluster randomization is honest but costs power.

In [ ]:
rho_values = [0.0, 0.05, 0.1, 0.2]
base_n = 400
m = 20  # sessions per cluster for illustration
rows = []
for rho in rho_values:
    # design effect for cluster randomization: 1 + (m-1)*rho
    deff = 1 + (m - 1) * rho
    n_needed = int(np.ceil(base_n * deff))
    rows.append({'rho': rho, 'design_effect': round(deff,2), 'users_needed_vs_rho0': n_needed})
print(pd.DataFrame(rows))
print('Higher intra-cluster correlation -> larger required sample for same power.')

## 6. What you do about it

- Pick the **randomization unit** to match how the treatment is delivered and how spillover travels (`U06-A5` - no lecture video; Kohavi ch. 14).
- If `SUTVA` fails at user level (`V16` ride-sharing), move to cluster or switchback designs.
- Expect **wider intervals** at cluster level - that is honesty, not failure.

**When this matters less:** Low interference, one session per user, treatment confined to one visit.

---

**Takeaway:** More rows is not more information if rows are dependent. Wrong units contaminate controls; cluster designs pay a power tax for honesty.

**Back to the unit:** [V1](../V1/units/unit-06-assignment-mechanism/README.md) · [V2](../V2/units/unit-06-assignment-mechanism/README.md)